In [ ]:
# Общие импорты и локальные данные JERICHON
import os, sys
from pathlib import Path

_HERE = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).expanduser().resolve()
_CANDIDATES = [_HERE, *_HERE.parents, Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((p for p in dict.fromkeys(_CANDIDATES)
                     if (p / "src" / "data" / "load_data.py").is_file()
                     and (p / "research").is_dir()), None)
if PROJECT_ROOT is None and os.environ.get("JERICHON_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["JERICHON_PROJECT_ROOT"]).expanduser().resolve()
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "data" / "load_data.py").is_file():
    raise FileNotFoundError("Не найден корень JERICHON с src/data/load_data.py")
sys.path[:0] = [str(PROJECT_ROOT), str(PROJECT_ROOT / "data")]
from src.data.load_data import *
ROOT = PROJECT_ROOT


# JERICHON — Super Candles: группировка признаков и event study v2

**Цель:** проверить источники и семейства Super Candle-признаков без 401 отдельных тестов.

Зафиксированные архитектурные правила из диагностики: Super Candles = только `tradestats`, `obstats`, `orderstats`; `orderbook` отдельно. Не делать простой join по `ts`: использовать существующий causal pipeline с валидированным `available_at <= bar_ts` и backward join. `SYSTIME` нельзя глобально считать UTC или MSK без проверки.

Этот notebook повторно строит локальный causal dataset, сворачивает трансформации признаков в корневые сигналы, формирует source/family scores и объединяет соседние срабатывания в эпизоды. На диск записывается **ровно один CSV**: `SC_02_group_research_summary.csv`. Результаты разведочные, не доказательство торгового преимущества.

## 1. Конфигурация

Используется локальная история AlgoPack, без API-запросов. Порог Dollar Bars должен совпадать с предыдущим исследованием.

In [1]:
from pathlib import Path
import sys, inspect, warnings, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKERS = ["SMLT", "GAZP", "MTLR"]
DOLLAR_THRESHOLD = 10_000_000
ROLLING_WINDOW = 500
MIN_HISTORY = 100
ANOMALY_QUANTILES = (0.95, 0.99)
GROUP_QUANTILES = (0.95, 0.99)
EPISODE_GAP_BARS = 3
MAX_CONTROLS_PER_EVENT = 3
RANDOM_STATE = 17
PROJECT_ROOT = ROOT
DATA_ROOT = PROJECT_ROOT / "data"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.builders import jerichon_feature_pipeline as jfp

required_helpers = ("build_dollar_bars_from_trades", "_base", "_build_sc_features", "_choose_availability", "_attach_source")
missing = [x for x in required_helpers if not hasattr(jfp, x)]
if missing:
    raise RuntimeError(f"В установленном jerichon_feature_pipeline.py отсутствуют helpers: {missing}")

print("Project root:", PROJECT_ROOT)
print("History root:", HISTORY_ROOT)
print("Pipeline:", Path(inspect.getfile(jfp)).resolve())
print("Tickers:", TICKERS)
print("Dollar threshold:", f"{DOLLAR_THRESHOLD:,}")


Project root: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON
History root: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\storage\history
Pipeline: C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py
Tickers: ['SMLT', 'GAZP', 'MTLR']
Dollar threshold: 10,000,000


## 2. Построение причинного набора

Повторно используем проверенную загрузку и существующий feature pipeline; контракт 73 + 401 = 474 проверяется до анализа.

In [2]:

def resolve_feature_frame(source, frames):
    if source in frames:
        return frames[source]
    matches = [k for k in frames if source.lower() in str(k).lower()]
    if len(matches) == 1:
        return frames[matches[0]]
    raise RuntimeError(f"Не удалось однозначно сопоставить {source}; доступны: {list(frames)}")

def build_ticker_dataset(ticker):
    data, saved_dates = load_local_history(ticker, HISTORY_ROOT)
    bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=DOLLAR_THRESHOLD)
    if bars is None or bars.empty:
        raise RuntimeError(f"{ticker}: Dollar Bar builder вернул пустой результат.")
    if "bar_ts" not in bars.columns:
        if "close_time" not in bars.columns:
            raise RuntimeError(f"{ticker}: в bars нет bar_ts/close_time; columns={list(bars.columns)}")
        bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
    else:
        bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)
    bars = bars.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

    # The pipeline creates many derived columns by repeated assignment. Its
    # fragmentation PerformanceWarnings are known and are summarized once,
    # rather than flooding the notebook output.
    perf_warning_count = 0
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", pd.errors.PerformanceWarning)
        base = jfp._base(bars.copy())
    perf_warning_count += sum(issubclass(w.category, pd.errors.PerformanceWarning) for w in captured)
    if isinstance(base, tuple):
        base = base[0]
    if not isinstance(base, pd.DataFrame):
        raise TypeError(f"{ticker}: _base не вернул DataFrame")
    if "bar_ts" not in base.columns:
        if "close_time" in base.columns:
            base = base.rename(columns={"close_time": "bar_ts"})
        elif isinstance(base.index, pd.DatetimeIndex):
            base = base.reset_index().rename(columns={base.index.name or "index": "bar_ts"})
        elif "ts" in base.columns:
            base = base.rename(columns={"ts": "bar_ts"})
        else:
            raise RuntimeError(f"{ticker}: не найден временной ключ в base features")
    base["bar_ts"] = pd.to_datetime(base["bar_ts"], errors="coerce", utc=True)
    base = base.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last")

    bars_sc = bars.copy()
    if "ts" not in bars_sc.columns:
        bars_sc["ts"] = bars_sc["bar_ts"]
    if "CLOSE" not in bars_sc.columns:
        close_col = next((c for c in bars_sc.columns if str(c).lower() == "close"), None)
        if close_col is None:
            raise RuntimeError(f"{ticker}: нет CLOSE/close в Dollar Bars")
        bars_sc["CLOSE"] = bars_sc[close_col]
    bars_sc["ts"] = pd.to_datetime(bars_sc["ts"], errors="coerce", utc=True)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", pd.errors.PerformanceWarning)
        sc_result = jfp._build_sc_features(
            data,
            bars_sc.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True),
        )
    perf_warning_count += sum(issubclass(w.category, pd.errors.PerformanceWarning) for w in captured)
    feature_frames = sc_result[0] if isinstance(sc_result, tuple) else sc_result
    if not isinstance(feature_frames, dict):
        raise TypeError(f"{ticker}: _build_sc_features должен вернуть dict")

    causal = bars.copy()
    for source in ("tradestats", "obstats", "orderstats"):
        # Validate source availability using the pipeline's established helper.
        checked = jfp._choose_availability(data[source].copy(), source)
        if isinstance(checked, tuple):
            checked = checked[0]
        if checked is None or len(checked) == 0:
            raise RuntimeError(f"{ticker}: availability validation вернула пустой {source}")
        ff = resolve_feature_frame(source, feature_frames)
        attached = jfp._attach_source(causal, ff, data[source], source)
        if isinstance(attached, pd.DataFrame):
            causal = attached

    base_cols_raw = [c for c in base.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base[c])]
    if len(base_cols_raw) != 73:
        raise RuntimeError(f"{ticker}: ожидалось 73 base features, найдено {len(base_cols_raw)}")
    base_join = base[["bar_ts"] + base_cols_raw].copy()
    base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols_raw})
    causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
    causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

    provenance = [c for c in causal.columns if c.endswith("__event_ts") or c.endswith("__available_at")]
    prefixes = ("tradestats__", "obstats__", "orderstats__")
    sc_cols = [c for c in causal.columns if c.startswith(prefixes) and c not in provenance and pd.api.types.is_numeric_dtype(causal[c])]
    # IMPORTANT: test the dtype of the SERIES causal[c], not the column-name
    # string c. Testing c silently dropped all 73 base features.
    base_feature_cols = [
        c for c in causal.columns
        if c.startswith("base__") and pd.api.types.is_numeric_dtype(causal[c])
    ]
    feature_cols = base_feature_cols + sc_cols

    # Validate each block separately so a mismatch reports exactly which block
    # is absent; don't silently skip a ticker or hide the root cause.
    if len(base_feature_cols) != len(base_cols_raw):
        raise RuntimeError(
            f"{ticker}: после merge сохранилось {len(base_feature_cols)} из "
            f"{len(base_cols_raw)} базовых признаков"
        )
    if not sc_cols:
        raise RuntimeError(
            f"{ticker}: после causal attach не найдено числовых SC-признаков. "
            f"Колонки итогового frame: {list(causal.columns)}"
        )
    if len(feature_cols) != 474:
        raise RuntimeError(
            f"{ticker}: состав признаков отличается от ожидаемого: "
            f"base={len(base_feature_cols)} (ожидалось 73), "
            f"SC={len(sc_cols)} (ожидалось 401), total={len(feature_cols)}. "
            "Тикер не будет молча отброшен: проверь названия колонок в pipeline."
        )
    causal = causal.sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)
    # Keep bar fields for forward path analysis, plus features and provenance.
    if perf_warning_count:
        print(f"{ticker}: подавлено предупреждений о фрагментации DataFrame: {perf_warning_count}")
    return {
        "ticker": ticker, "data": data, "dates": saved_dates, "bars": bars,
        "dataset": causal, "feature_cols": feature_cols, "base_cols": base_feature_cols,
        "sc_cols": sc_cols, "provenance": provenance,
    }

datasets = {}
load_rows = []
load_errors = []
for ticker in TICKERS:
    try:
        obj = build_ticker_dataset(ticker)
        datasets[ticker] = obj
        load_rows.append({
            "ticker": ticker, "saved_dates": len(obj["dates"]),
            "date_start": obj["dates"][0], "date_end": obj["dates"][-1],
            "dollar_bars": len(obj["bars"]), "base_features": len(obj["base_cols"]),
            "supercandle_features": len(obj["sc_cols"]), "features_total": len(obj["feature_cols"]),
        })
        print(
            f"OK {ticker}: bars={len(obj['bars']):,}; "
            f"base={len(obj['base_cols'])}; SC={len(obj['sc_cols'])}; "
            f"dates={obj['dates'][0]} → {obj['dates'][-1]}"
        )
    except Exception as e:
        message = f"{ticker}: {type(e).__name__}: {e}"
        load_errors.append(message)
        print(f"ERROR {message}")
display(pd.DataFrame(load_rows))
if load_errors:
    print("\nОшибки загрузки (кратко):")
    for message in load_errors:
        print(" -", message)
if not datasets:
    raise RuntimeError(
        "Не удалось подготовить данные ни для одного тикера. Причины выше: "
        + " | ".join(load_errors)
    )


SMLT: подавлено предупреждений о фрагментации DataFrame: 74
OK SMLT: bars=2,672; base=73; SC=401; dates=2026-09-25 → 2026-10-20
GAZP: подавлено предупреждений о фрагментации DataFrame: 74
OK GAZP: bars=9,993; base=73; SC=401; dates=2026-09-24 → 2026-10-20
MTLR: подавлено предупреждений о фрагментации DataFrame: 74
OK MTLR: bars=1,492; base=73; SC=401; dates=2026-08-28 → 2026-10-20


,ticker,saved_dates,date_start,date_end,dollar_bars,base_features,supercandle_features,features_total
0,SMLT,14,2026-09-25,2026-10-20,2672,73,401,474
1,GAZP,15,2026-09-24,2026-10-20,9993,73,401,474
2,MTLR,33,2026-08-28,2026-10-20,1492,73,401,474


## 3. Свернуть 401 столбец в корневые метрики и смысловые семьи

Например, `x`, `x_ma3`, `x_z12`, `x_chg`, `x_ret` относятся к одному корневому сигналу. Семьи по названиям эвристические и нужны только для первого анализа.

In [3]:
import math, re
from collections import defaultdict

TRANSFORM_RE = re.compile(r"_(?:ma\d+|z\d+|chg|ret)$", re.IGNORECASE)
def root_feature_name(col):
    source, metric = col.split("__", 1)
    while True:
        new_metric = TRANSFORM_RE.sub("", metric)
        if new_metric == metric: break
        metric = new_metric
    return f"{source}__{metric}"
def transform_complexity(col):
    return len(TRANSFORM_RE.findall(col.split("__", 1)[1]))
def semantic_family(root_col):
    m = root_col.split("__", 1)[1].lower()
    if any(k in m for k in ("imb", "imbalance", "bid", "ask", "buy", "sell", "aggress", "b_s", "share")): return "imbalance_pressure"
    if any(k in m for k in ("spread", "liquidity", "cost", "vwap", "px_", "price", "mid", "microprice", "bps")): return "liquidity_price"
    if any(k in m for k in ("volume", "value", "turnover", "trade", "count", "qty", "notional", "amount")): return "activity_volume"
    if any(k in m for k in ("order", "depth", "level", "queue", "cancel", "add", "modify")): return "orders_depth"
    return "other"

# Inventory is shared across tickers; the loader separately validates 73 + 401 = 474 per ticker.
all_sc_cols = datasets[next(iter(datasets))]["sc_cols"]
root_members = defaultdict(list)
for col in all_sc_cols: root_members[root_feature_name(col)].append(col)
root_rows = []
for root, members in sorted(root_members.items()):
    source = root.split("__", 1)[0]
    chosen = root if root in members else sorted(members, key=lambda c:(transform_complexity(c),len(c),c))[0]
    root_rows.append({"root":root,"source":source,"family":semantic_family(root),"chosen_feature":chosen,"variants":len(members),"is_raw_root":chosen==root})
root_map = pd.DataFrame(root_rows)
print(f"401 SC columns collapsed into {root_map.root.nunique()} root metrics")
print("Root metrics by source/family:")
print(root_map.groupby(["source","family"]).size().rename("roots").to_string())
print("\nTransformation variants per root:")
print(root_map.variants.value_counts().sort_index().rename_axis("variants").to_string())


401 SC columns collapsed into 51 root metrics
Root metrics by source/family:
source      family            
obstats     activity_volume        2
            imbalance_pressure     7
            liquidity_price        7
orderstats  activity_volume        4
            imbalance_pressure    13
            liquidity_price        2
            orders_depth           2
tradestats  activity_volume        2
            imbalance_pressure    10
            other                  2

Transformation variants per root:
variants
1     16
11    35


## 4. Причинные групповые scores

Для каждого корневого признака отклонение от rolling median нормируется на прошлый 95-й перцентиль отклонений. Все окна сдвинуты на один бар. Затем групповой score — среднее верхних 20% root-scores; событие — превышение прошлого 95-го или 99-го перцентиля группового score.

In [4]:
def find_ohlc_columns(bars):
    lower = {str(c).lower():c for c in bars.columns}
    out = {k:next((lower[x] for x in names if x in lower),None) for k,names in {
        "open":("open","o"),"high":("high","h"),"low":("low","l"),"close":("close","c")}.items()}
    missing = [k for k,v in out.items() if v is None]
    if missing: raise RuntimeError(f"OHLC columns missing {missing}; columns={list(bars.columns)}")
    return out

def root_scores_for_ticker(obj):
    ds = obj["dataset"].reset_index(drop=True)
    out = {}
    for _, r in root_map.iterrows():
        col = r["chosen_feature"]
        if col not in ds: continue
        x = pd.to_numeric(ds[col],errors="coerce").replace([np.inf,-np.inf],np.nan)
        med = x.rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).median().shift(1)
        dev = (x-med).abs()
        scale = dev.rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).quantile(.95).shift(1)
        out[r["root"]] = dev / scale.where(scale>0)
    return pd.DataFrame(out)

def top_fraction_mean(frame, fraction=.20):
    if frame.shape[1]==0: return pd.Series(np.nan,index=frame.index)
    a=frame.to_numpy(dtype=float).copy(); a[~np.isfinite(a)]=-np.inf; a.sort(axis=1)
    k=max(1,int(math.ceil(frame.shape[1]*fraction))); top=a[:,-k:]
    ok=np.isfinite(top); sums=np.where(ok,top,0).sum(axis=1); counts=ok.sum(axis=1)
    return pd.Series(np.divide(sums,counts,out=np.full(len(frame),np.nan),where=counts>0),index=frame.index)

def group_scores_for_ticker(obj):
    roots=root_scores_for_ticker(obj); meta=root_map.set_index("root"); groups={}
    for source in ("tradestats","obstats","orderstats"):
        cols=[c for c in roots if meta.loc[c,"source"]==source]
        if cols: groups[f"source::{source}"]=top_fraction_mean(roots[cols])
    for (source,family),part in root_map.groupby(["source","family"]):
        cols=[c for c in part.root.tolist() if c in roots]
        if len(cols)>=2: groups[f"source_family::{source}::{family}"]=top_fraction_mean(roots[cols])
    scores=pd.DataFrame(groups)
    flags={}; thresholds={}
    for col in scores:
        for q in GROUP_QUANTILES:
            th=scores[col].rolling(ROLLING_WINDOW,min_periods=MIN_HISTORY).quantile(q).shift(1)
            thresholds[(col,q)]=th; flags[(col,q)]=scores[col].gt(th)&th.notna()&scores[col].notna()
    return scores,thresholds,flags

group_results={}
for ticker,obj in datasets.items():
    scores,thresholds,flags=group_scores_for_ticker(obj)
    group_results[ticker]={"scores":scores,"thresholds":thresholds,"flags":flags}
    print(f"{ticker}: {scores.shape[1]} source/family groups; {len(scores):,} bars")


SMLT: 13 source/family groups; 2,672 bars
GAZP: 13 source/family groups; 9,993 bars
MTLR: 13 source/family groups; 1,492 bars


# SC_04 — поиск частых торговых возможностей на движениях 0,25–1%

## Цель

Предыдущее исследование показало, что длинные горизонты насыщаются: на MTLR движение 0,5% за 5–20 Dollar Bars происходит почти всегда и в событиях, и в контроле. Поэтому для цели «около 5 сделок в день на инструмент» важнее короткий горизонт, частота сигналов и отличие от обычного рынка.

В этом исследовании:
- проверяются пороги группового score от 80-го до 99-го перцентиля;
- отдельно измеряется число сигналов в день при разных расстояниях между повторными сигналами;
- проверяются движения 0,25%, 0,5%, 0,75% и 1% вверх, вниз и в любую сторону;
- основное внимание — горизонты 1, 2, 3 и 5 Dollar Bars;
- контроль для каждого сигнала берётся из того же торгового дня, из баров, не находящихся рядом с сигналами. Это проще и прозрачнее прежнего matching, но не устраняет все различия режимов.

**Важно:** это ещё не торговый бэктест. Мы пока оцениваем, даёт ли сигнал достаточно частую и полезную возможность для входа. Спред, комиссии, проскальзывание, исполнимая цена и порядок касания TP/SL внутри OHLC-бара будут проверяться следующим шагом.

In [6]:
# Исследовательские параметры
QUANTILES_TO_TEST = (0.80, 0.85, 0.90, 0.95, 0.99)
HORIZONS_FAST = (1, 2, 3, 5, 10)
MOVE_LEVELS_PCT = (0.25, 0.50, 0.75, 1.00)
SIGNAL_GAPS = (0, 1, 2)  # минимум баров между выбранными сигналами; 0 = все срабатывания
MIN_DAILY_HISTORY = 1
OUTPUT_CSV = Path("research/supercandles/mark_4/results/SC_04_frequency_and_small_moves.csv"); OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print("Проверяем квантили:", QUANTILES_TO_TEST)
print("Горизонты:", HORIZONS_FAST, "Dollar Bars")
print("Пороги движения:", MOVE_LEVELS_PCT, "%")
print("Варианты разрежения сигналов:", SIGNAL_GAPS)

Проверяем квантили: (0.8, 0.85, 0.9, 0.95, 0.99)
Горизонты: (1, 2, 3, 5, 10) Dollar Bars
Пороги движения: (0.25, 0.5, 0.75, 1.0) %
Варианты разрежения сигналов: (0, 1, 2)


## 1. Сначала частота, а не статистическая значимость

Один и тот же участок может вызвать несколько групповых сигналов почти одновременно. Поэтому считаем:
- сигналы каждой группы отдельно;
- объединённый сигнал `ANY_GROUP` — сработала хотя бы одна группа;
- количество исходных срабатываний и количество после разрежения сигналов;
- среднее, медианное и 90-й перцентиль числа сигналов в торговый день.

Цель 5 сделок в день проверяем именно по числу возможностей в день, а не по доле сигналов среди баров.

In [7]:
SESSION_TIMEZONE = "Europe/Moscow"  # изменить только если bar_ts в проекте не соответствует UTC-нормализации

def get_bar_dates(bars):
    col = "bar_ts" if "bar_ts" in bars.columns else ("close_time" if "close_time" in bars.columns else None)
    if col is None:
        raise RuntimeError(f"Не найден bar_ts/close_time. Колонки: {list(bars.columns)}")
    # В сборщике текущего пайплайна bar_ts приводится к UTC; для подсчёта сделок в день
    # переводим время в биржевой часовой пояс. Не применять к сырым SYSTIME без проверки.
    ts = pd.to_datetime(bars[col], errors="coerce", utc=True).dt.tz_convert(SESSION_TIMEZONE)
    return ts.dt.strftime("%Y-%m-%d").fillna("UNKNOWN").to_numpy(), ts

def thin_events(mask, min_gap):
    idx = np.flatnonzero(np.asarray(mask, dtype=bool))
    if min_gap <= 0:
        return idx.tolist()
    chosen, last = [], -10**9
    for i in idx:
        if i - last > min_gap:
            chosen.append(int(i))
            last = int(i)
    return chosen

def future_path_flags(close, high, low, i, horizon, threshold):
    p0 = close[i]
    j = i + horizon
    if j >= len(close) or not np.isfinite(p0) or p0 <= 0:
        return None
    hh = high[i+1:j+1]
    ll = low[i+1:j+1]
    hh = hh[np.isfinite(hh)]
    ll = ll[np.isfinite(ll)]
    if len(hh) == 0 or len(ll) == 0:
        return None
    up = (np.max(hh) / p0 - 1) * 100
    down = (np.min(ll) / p0 - 1) * 100
    return {
        "up": float(up >= threshold),
        "down": float(down <= -threshold),
        "either": float(max(abs(up), abs(down)) >= threshold),
        "up_excursion_pct": float(up),
        "down_excursion_pct": float(down),
        "close_return_pct": float((close[j] / p0 - 1) * 100) if np.isfinite(close[j]) else np.nan,
    }

def daily_count_stats(indices, dates):
    by_day = pd.Series([dates[i] for i in indices]).value_counts()
    # Дни без сигналов тоже важны: учитываем все даты в наборе ниже при агрегации.
    all_days = pd.Index(pd.unique(dates))
    by_day = by_day.reindex(all_days, fill_value=0)
    return {
        "days": int(len(all_days)),
        "signals_total": int(len(indices)),
        "signals_per_day_mean": float(by_day.mean()) if len(by_day) else np.nan,
        "signals_per_day_median": float(by_day.median()) if len(by_day) else np.nan,
        "signals_per_day_p90": float(by_day.quantile(.90)) if len(by_day) else np.nan,
        "days_ge_5_signals_pct": float((by_day >= 5).mean() * 100) if len(by_day) else np.nan,
        "days_ge_3_signals_pct": float((by_day >= 3).mean() * 100) if len(by_day) else np.nan,
    }

## 2. Считаем частоту срабатываний и результаты короткого горизонта

Порог группового score каждый раз рассчитывается по прошлым данным с `shift(1)`: текущий бар не участвует в расчёте собственного порога. Это сохраняет причинность.

Контроль для события — бары того же календарного дня, которые не являются сигналами и отстоят от выбранных событий дальше максимального исследуемого горизонта. Вероятность движения в контроле считается по таким барам. Сравнение внутри дня помогает не смешивать, например, спокойные и кризисные дни.

У событий и контроля окна будущих исходов могут перекрываться друг с другом; поэтому цифры ниже — диагностические оценки эффекта, а не формальные доказательства значимости. Основной критерий следующего этапа — повторяемость эффекта на отложенных датах.

In [8]:
frequency_rows = []
outcome_rows = []

for ticker, obj in datasets.items():
    bars = obj["bars"].reset_index(drop=True)
    ohlc = find_ohlc_columns(bars)
    close = pd.to_numeric(bars[ohlc["close"]], errors="coerce").to_numpy(float)
    high = pd.to_numeric(bars[ohlc["high"]], errors="coerce").to_numpy(float)
    low = pd.to_numeric(bars[ohlc["low"]], errors="coerce").to_numpy(float)
    dates, timestamps = get_bar_dates(bars)
    n = len(bars)
    valid_end = n - max(HORIZONS_FAST) - 1
    if valid_end <= MIN_HISTORY:
        print(f"WARNING {ticker}: недостаточно баров")
        continue

    scores = group_results[ticker]["scores"].reset_index(drop=True)
    # Отдельные группы + объединённый сигнал: хотя бы одна группа выше своего причинного порога.
    group_masks = {}
    for col in scores.columns:
        for q in QUANTILES_TO_TEST:
            threshold = scores[col].rolling(ROLLING_WINDOW, min_periods=MIN_HISTORY).quantile(q).shift(1)
            mask = scores[col].gt(threshold) & threshold.notna() & scores[col].notna()
            m = mask.to_numpy(dtype=bool)
            m[valid_end:] = False
            group_masks[(col, q)] = m

    for q in QUANTILES_TO_TEST:
        union = np.zeros(n, dtype=bool)
        for (col, qq), mask in group_masks.items():
            if qq == q:
                union |= mask
        analysis_masks = [(col, qq, mask) for (col, qq), mask in group_masks.items() if qq == q]
        analysis_masks.append(("ANY_GROUP", q, union))

        for group_key, qq, raw_mask in analysis_masks:
            group_type, group_name = ("union", "ANY_GROUP") if group_key == "ANY_GROUP" else tuple(group_key.split("::", 1))
            raw_indices = np.flatnonzero(raw_mask).tolist()
            for gap in SIGNAL_GAPS:
                selected = thin_events(raw_mask, gap)
                fstats = daily_count_stats(selected, dates)
                frequency_rows.append({
                    "row_type": "frequency", "ticker": ticker, "group_type": group_type,
                    "group": group_name, "quantile": q, "signal_gap_bars": gap,
                    "bars_total": n, "bars_valid_for_outcome": valid_end,
                    "raw_signal_bars": int(raw_mask.sum()),
                    "raw_signal_rate_pct": 100 * raw_mask.sum() / max(valid_end, 1),
                    "selected_signals": len(selected),
                    **fstats,
                })

                # Outcome is evaluated on the selected signals; do not require a large 1.5-2% move.
                for h in HORIZONS_FAST:
                    events = []
                    for i in selected:
                        m = future_path_flags(close, high, low, i, h, 0.5)
                        if m is not None:
                            events.append((i, m))
                    if not events:
                        continue
                    event_set = set(selected)
                    # Clean same-day controls: not a signal and not close to a selected signal.
                    controls_by_day = {}
                    for i in range(MIN_HISTORY, valid_end):
                        if raw_mask[i]:
                            continue
                        if any(abs(i - e) <= max(HORIZONS_FAST) for e in selected):
                            continue
                        controls_by_day.setdefault(dates[i], []).append(i)

                    # Precompute all threshold outcomes for event and control bars.
                    for threshold in MOVE_LEVELS_PCT:
                        for side in ("up", "down", "either"):
                            ev_vals, ctrl_vals = [], []
                            for i, _ in events:
                                result = future_path_flags(close, high, low, i, h, threshold)
                                if result is not None:
                                    ev_vals.append((i, result[side]))
                            # Same-day baseline: average control-bar probability per event day.
                            per_event_diffs = []
                            event_probs, control_probs = [], []
                            for i, val in ev_vals:
                                candidates = controls_by_day.get(dates[i], [])
                                cv = []
                                for j in candidates:
                                    rr = future_path_flags(close, high, low, j, h, threshold)
                                    if rr is not None:
                                        cv.append(rr[side])
                                if cv:
                                    event_probs.append(val)
                                    control_probs.append(float(np.mean(cv)))
                                    per_event_diffs.append(val - float(np.mean(cv)))
                            if not per_event_diffs:
                                continue
                            outcome_rows.append({
                                "row_type": "outcome", "ticker": ticker, "group_type": group_type,
                                "group": group_name, "quantile": q, "signal_gap_bars": gap,
                                "horizon_bars": h, "threshold_pct": threshold, "side": side,
                                "episodes_or_signals": len(event_probs),
                                "event_probability_pct": 100 * float(np.mean(event_probs)),
                                "same_day_control_probability_pct": 100 * float(np.mean(control_probs)),
                                "difference_pp": 100 * float(np.mean(per_event_diffs)),
                                "median_event_minus_control_pp": 100 * float(np.median(per_event_diffs)),
                            })

freq_df = pd.DataFrame(frequency_rows)
outcome_df = pd.DataFrame(outcome_rows)
if freq_df.empty:
    raise RuntimeError("Частотные результаты пусты: проверь datasets/group_results.")
combined = pd.concat([freq_df, outcome_df], ignore_index=True, sort=False)
combined.to_csv(OUTPUT_CSV, index=False)
print(f"Сохранён единый CSV: {OUTPUT_CSV.resolve()}")
print("Строк частоты:", len(freq_df), "| строк исходов:", len(outcome_df))
display(freq_df[(freq_df.group == "ANY_GROUP")].sort_values(["ticker","quantile","signal_gap_bars"]).head(40))

KeyboardInterrupt: 

## 3. Как выбрать кандидатов для следующего этапа

Сначала отфильтруй `row_type = frequency`, `group = ANY_GROUP`. Смотри:
- `signals_per_day_mean` — среднее число возможностей в день;
- `signals_per_day_median` — типичный день, менее чувствительный к всплескам;
- `days_ge_5_signals_pct` — процент дней, в которые было не менее пяти сигналов;
- `signal_gap_bars` — насколько мы подавляем почти дублирующие сигналы.

Затем для тех же тикеров/квантилей смотри `row_type = outcome`, прежде всего `threshold_pct = 0.5`, горизонты 1–3 бара и `side = up/down/either`.

- `difference_pp = +8` означает, что движение заданного размера встречалось после сигнала примерно на 8 процентных пунктов чаще, чем в контрольных барах того же дня.
- Положительное значение `either` говорит о связи с движением в любую сторону; положительные `up` и отрицательные/положительные `down` нужно читать отдельно, поскольку `down` здесь тоже кодирует вероятность достижения порога падения, а не знак доходности.
- Если частота достигает пяти сигналов в день, но вероятность движения 0,5% почти не отличается от контроля, группа пока не доказала полезность.
- Если эффект есть только при редких сигналах, это конфликтует с целевой частотой и требует проверки компромисса между частотой и качеством.

Следующий шаг после этого sweep: выбрать несколько кандидатов по заранее заданному правилу и проверить их на хронологически отложенном периоде. Только потом моделировать вход на следующей доступной цене, спред, комиссии, проскальзывание и выход по TP/SL. Порог 0,5% — это движение цены до расходов, не чистая прибыль.